# Import Statements

In [1]:
import os
import glob
import shutil
import cv2
from PIL import Image
from matplotlib import pyplot as plt
import random
from ultralytics import YOLO

# Check and Set Working Directory

In [2]:
pwd

'/home/exh4748/ProjectTortoise/Beta'

In [3]:
# Setting base directory path
base_dir = "/home/exh4748/ProjectTortoise/ObjectData/data2"
os.chdir(base_dir)
base_dir = os.getcwd()

In [4]:
# List the contents of train, val, and test folders
train_contents = os.listdir(os.path.join(base_dir, "train"))
val_contents = os.listdir(os.path.join(base_dir, "val"))
test_contents = os.listdir(os.path.join(base_dir, "test"))

print("Train Folder Contents:", train_contents)
print("Validation Folder Contents:", val_contents)
print("Test Folder Contents:", test_contents)

Train Folder Contents: ['cellphone.cache', 'labels.cache', 'images', '.ipynb_checkpoints', 'labels']
Validation Folder Contents: ['cellphone.cache', 'labels.cache', 'images', '.ipynb_checkpoints', 'labels']
Test Folder Contents: ['images', '.ipynb_checkpoints', 'labels']


# Setting Up Dataset For Model

In [5]:
import tensorflow as tf

# Check if any GPU is detected
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  4


In [6]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Set TensorFlow to use only the first GPU
        tf.config.set_visible_devices(gpus[0], 'GPU')
        print(f"Using GPU: {gpus[0]}")
    except RuntimeError as e:
        print(e)

Using GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [7]:
datasets = ["train", "val", "test"]

# loading YOLOv8 model (pre-trained on COCO dataset)
model = YOLO("yolov8n.pt")
# Map COCO class names to your custom dataset classes
coco_classes = model.names  # Pre-trained COCO class labels
custom_classes = ["cellphone", "wallet", "watch"]
custom_class_ids = {coco_classes.index(cls): i for i, cls in enumerate(custom_classes) if cls in coco_classes}

In [8]:
# Define base directory
base_dir = "/home/exh4748/ProjectTortoise/ObjectData/data2"
datasets = ["train", "val", "test"]
categories = ["cellphone", "wallet", "watch"]

# Loop through train, val, test
for dataset in datasets:
    dataset_path = os.path.join(base_dir, dataset)

    # Create unified images and labels directories
    images_path = os.path.join(dataset_path, "images")
    labels_path = os.path.join(dataset_path, "labels")

    os.makedirs(images_path, exist_ok=True)
    os.makedirs(labels_path, exist_ok=True)

    # Move images and labels from category subdirectories
    for category in categories:
        category_path = os.path.join(dataset_path, category)
        
        if not os.path.exists(category_path):
            continue  # Skip if category folder doesn't exist

        # Move images
        image_files = glob.glob(os.path.join(category_path, "*.jpg")) + \
                      glob.glob(os.path.join(category_path, "*.png")) + \
                      glob.glob(os.path.join(category_path, "*.jpeg"))

        for image_file in image_files:
            filename = os.path.basename(image_file)
            new_image_path = os.path.join(images_path, filename)
            shutil.move(image_file, new_image_path)

        # Move corresponding labels
        label_files = glob.glob(os.path.join(category_path, "*.txt"))

        for label_file in label_files:
            filename = os.path.basename(label_file)
            new_label_path = os.path.join(labels_path, filename)
            shutil.move(label_file, new_label_path)

        # Remove empty category folders
        shutil.rmtree(category_path, ignore_errors=True)

print("✅ Dataset successfully reorganized for YOLO training!")

✅ Dataset successfully reorganized for YOLO training!


In [9]:
# Number of images to keep
num_images_to_keep = 5000

# Loop through each dataset (train, val, test)
for dataset in ["train", "val", "test"]:
    dataset_path = os.path.join(base_dir, dataset)
    images_path = os.path.join(dataset_path, "images")
    labels_path = os.path.join(dataset_path, "labels")

    # Get all images in the dataset
    image_files = glob.glob(os.path.join(images_path, "*.jpg")) + \
                  glob.glob(os.path.join(images_path, "*.png")) + \
                  glob.glob(os.path.join(images_path, "*.jpeg"))

    # Ensure there are enough images
    if len(image_files) < num_images_to_keep:
        print(f"🚨 Not enough images in {dataset}. Found {len(image_files)} images.")
        continue

    # Randomly select 500 images
    selected_images = random.sample(image_files, num_images_to_keep)

    # Create a new folder to store reduced dataset
    reduced_images_path = os.path.join(base_dir, f"{dataset}_reduced/images")
    reduced_labels_path = os.path.join(base_dir, f"{dataset}_reduced/labels")
    os.makedirs(reduced_images_path, exist_ok=True)
    os.makedirs(reduced_labels_path, exist_ok=True)

    # Move selected images and their corresponding labels
    for img_path in selected_images:
        filename = os.path.basename(img_path)
        
        # Move image
        shutil.move(img_path, os.path.join(reduced_images_path, filename))

        # Move corresponding label file (if exists)
        label_path = img_path.replace("images", "labels").replace(".jpg", ".txt").replace(".png", ".txt").replace(".jpeg", ".txt")
        if os.path.exists(label_path):
            shutil.move(label_path, os.path.join(reduced_labels_path, os.path.basename(label_path)))

    print(f"✅ {dataset} reduced to {num_images_to_keep} images.")

print("✅ Dataset reduction completed!")

✅ train reduced to 5000 images.
✅ val reduced to 5000 images.
✅ test reduced to 5000 images.
✅ Dataset reduction completed!


In [10]:
# Load YOLO model
model = YOLO("yolov8n.pt")  # Change to 'yolov8s.pt' or 'yolov8m.pt' for better accuracy

# Process reduced datasets
for dataset in ["train_reduced", "val_reduced", "test_reduced"]:
    image_folder = os.path.join(base_dir, dataset, "images")
    label_folder = os.path.join(base_dir, dataset, "labels")
    os.makedirs(label_folder, exist_ok=True)

    image_files = glob.glob(f"{image_folder}/*.jpg") + glob.glob(f"{image_folder}/*.png")

    print(f"Processing {len(image_files)} images in {dataset}...")

    for image_path in image_files:
        results = model(image_path)
        label_path = os.path.join(label_folder, os.path.basename(image_path).replace(".jpg", ".txt").replace(".png", ".txt"))

        with open(label_path, "w") as f:
            for result in results:
                for box in result.boxes:
                    class_id = int(box.cls[0].item())
                    x_center, y_center, width, height = box.xywh[0].cpu().numpy()
                    f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

    print(f"✅ Labels successfully generated for {dataset}!")

print("✅ Relabeling completed!")

Processing 5281 images in train_reduced...

image 1/1 /home/exh4748/ProjectTortoise/ObjectData/data2/train_reduced/images/aug_89104_Q7W390GLJDMR.jpg: 640x640 (no detections), 4.7ms
Speed: 3.7ms preprocess, 4.7ms inference, 8.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/exh4748/ProjectTortoise/ObjectData/data2/train_reduced/images/aug_83787_5G6RTJNJYBSA.jpg: 640x640 (no detections), 4.3ms
Speed: 1.4ms preprocess, 4.3ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/exh4748/ProjectTortoise/ObjectData/data2/train_reduced/images/aug_44202_2TCZOS5GKB3T.jpg: 640x640 (no detections), 4.0ms
Speed: 0.8ms preprocess, 4.0ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/exh4748/ProjectTortoise/ObjectData/data2/train_reduced/images/aug_58705_LWSCO7BAFG7D.jpg: 640x640 (no detections), 5.4ms
Speed: 0.9ms preprocess, 5.4ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/exh4

# Debugging and Verifying Everything is Good to Go

In [11]:
# Function to print directory structure
def print_directory_structure(base_path):
    for root, dirs, files in os.walk(base_path):
        level = root.replace(base_path, "").count(os.sep)
        indent = " " * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = " " * 4 * (level + 1)
        for f in files[:5]:  # Print only first 5 files for brevity
            print(f"{sub_indent}{f}")
        if len(files) > 5:
            print(f"{sub_indent}... ({len(files)} files)")

print("📂 Checking dataset structure...")
print_directory_structure(base_dir)

📂 Checking dataset structure...
data2/
    yolo11n.pt
    data.yaml
    yolov8n.pt
    train/
        cellphone.cache
        labels.cache
        images/
            aug_20871_02NYXWWEYUN4.jpg
            aug_50499_012_0bef5f6f.jpg
            aug_92117_3RIPGFWFSZEP.jpg
            aug_34910_079_fe404f1e.jpg
            aug_31200_067_da385cad.jpg
            ... (173234 files)
        .ipynb_checkpoints/
        labels/
            aug_60534_08B7ICIBYXXJ.txt
            aug_25896_078_6b3ec48a.txt
            aug_47165_M0AUGBFOCQ1C.txt
            aug_71017_DILLXNFXLDVV.txt
            aug_71519_9WQRYO5MRFZY.txt
            ... (173234 files)
    test_reduced/
        images/
            aug_86728_TN1QSZ8NHP48.jpg
            aug_6033_W3KX8PE7CGVW.jpg
            aug_9548_L99HRUXTFUAJ.jpg
            aug_83971_131_0bf3b860.jpg
            aug_28623_PWW7RSB4HDJU.jpg
            ... (5253 files)
        labels/
            aug_47783_W7M90HB8NP7Y.txt
            aug_80747_P5E42KLGXV34.txt

In [12]:
def count_images_and_labels(base_dir, dataset):
    image_files = glob.glob(os.path.join(base_dir, dataset, "images", "*"))
    label_files = glob.glob(os.path.join(base_dir, dataset, "labels", "*"))

    print(f"📊 {dataset}: {len(image_files)} images, {len(label_files)} labels")
    return image_files, label_files

# Check for train, val, and test sets
train_images, train_labels = count_images_and_labels(base_dir, "train_reduced")
val_images, val_labels = count_images_and_labels(base_dir, "val_reduced")
test_images, test_labels = count_images_and_labels(base_dir, "test_reduced")

# Ensure each image has a corresponding label
missing_labels = [img for img in train_images if not os.path.exists(img.replace("images", "labels").replace(".jpg", ".txt").replace(".png", ".txt"))]

if missing_labels:
    print(f"🚨 Warning: {len(missing_labels)} images in train_reduced have no labels!")
else:
    print("✅ All train images have corresponding labels.")


📊 train_reduced: 5281 images, 5281 labels
📊 val_reduced: 5296 images, 5296 labels
📊 test_reduced: 5253 images, 5253 labels
✅ All train images have corresponding labels.


In [13]:
# Pick a random image
random_image_path = random.choice(train_images)
random_label_path = random_image_path.replace("images", "labels").replace(".jpg", ".txt").replace(".png", ".txt")

print(f"🖼️ Random Image: {random_image_path}")
print(f"📄 Corresponding Label: {random_label_path}")

# Print label contents
if os.path.exists(random_label_path):
    with open(random_label_path, "r") as f:
        print("\nLabel Contents:\n", f.read())
else:
    print("🚨 Label file is missing!")

🖼️ Random Image: /home/exh4748/ProjectTortoise/ObjectData/data2/train_reduced/images/aug_89485_VOT3DPICZQTI.jpg
📄 Corresponding Label: /home/exh4748/ProjectTortoise/ObjectData/data2/train_reduced/labels/aug_89485_VOT3DPICZQTI.txt

Label Contents:
 0 234.6785888671875 270.3529968261719 284.68133544921875 208.7930145263672

